# FinQA Dataset Analysis

Exploratory analysis across all splits: train, dev, test, private_test.

In [13]:
import json
import re
import sys
from pathlib import Path
from collections import Counter

DATA_DIR = Path("../data")
SPLITS = ["train", "dev", "test", "private_test"]

# Make project imports available
sys.path.insert(0, str(Path("..").resolve()))
from finqa_chatbot.dsl.parser import parse_program_to_tokens
from finqa_chatbot.dsl.executor import eval_program

datasets = {}
for split in SPLITS:
    path = DATA_DIR / f"{split}.json"
    if path.exists():
        with open(path) as f:
            datasets[split] = json.load(f)
        print(f"{split}: {len(datasets[split])} entries")
    else:
        print(f"{split}: NOT FOUND")

train: 6251 entries
dev: 883 entries
test: 1147 entries
private_test: 919 entries


## exe_ans field analysis

In [14]:
for split, data in datasets.items():
    type_counts = Counter()
    str_values = Counter()
    missing = 0
    for entry in data:
        val = entry["qa"].get("exe_ans")
        if val is None:
            missing += 1
            continue
        t = type(val).__name__
        type_counts[t] += 1
        if isinstance(val, str):
            str_values[val] += 1

    print(f"\n=== {split} ({len(data)} entries) ===")
    if missing:
        print(f"  No exe_ans field: {missing}")
    if type_counts:
        print("Type distribution:")
        for t, c in type_counts.most_common():
            print(f"  {t}: {c} ({c/len(data):.1%})")
    if str_values:
        print("String values:")
        for v, c in str_values.most_common():
            print(f"  {repr(v)}: {c}")


=== train (6251 entries) ===
Type distribution:
  float: 6127 (98.0%)
  str: 124 (2.0%)
String values:
  'yes': 77
  'no': 47

=== dev (883 entries) ===
Type distribution:
  float: 873 (98.9%)
  str: 10 (1.1%)
String values:
  'no': 7
  'yes': 3

=== test (1147 entries) ===
Type distribution:
  float: 1127 (98.3%)
  str: 20 (1.7%)
String values:
  'yes': 13
  'no': 7

=== private_test (919 entries) ===
  No exe_ans field: 919


## Program field analysis

Identify all DSL operations, count valid vs invalid programs per split.

In [15]:
OP_RE = re.compile(r"([a-z_]+)\s*\(")

for split, data in datasets.items():
    op_counts = Counter()
    valid = 0
    invalid = 0
    no_program = 0
    invalid_examples = []

    for entry in data:
        prog_str = entry["qa"].get("program", "")
        if not prog_str:
            no_program += 1
            continue

        # Extract operation names
        ops = OP_RE.findall(prog_str)
        for op in ops:
            op_counts[op] += 1

        # Check syntax validity via parser + executor
        try:
            tokens = parse_program_to_tokens(prog_str)
            table = entry.get("table", [])
            inv_flag, result = eval_program(tokens, table)
            if inv_flag:
                invalid += 1
                if len(invalid_examples) < 5:
                    invalid_examples.append((entry["id"], prog_str, "executor_invalid"))
            else:
                valid += 1
        except Exception as e:
            invalid += 1
            if len(invalid_examples) < 5:
                invalid_examples.append((entry["id"], prog_str, str(e)))

    total = len(data)
    print(f"=== {split} ({total} entries) ===")
    print(f"  Valid:      {valid}")
    print(f"  Invalid:    {invalid}")
    print(f"  No program: {no_program}")
    print(f"  Operations ({len(op_counts)} unique):")
    for op, c in op_counts.most_common():
        print(f"    {op:<20s} {c:>5}")
    if invalid_examples:
        print(f"  Sample invalid programs:")
        for eid, prog, reason in invalid_examples:
            print(f"    {eid}: {prog[:80]}")
            print(f"      reason: {reason}")
    print()

=== train (6251 entries) ===
  Valid:      6251
  Invalid:    0
  No program: 0
  Operations (10 unique):
    divide                4445
    subtract              2739
    add                   1512
    multiply               567
    greater                124
    table_average           95
    table_max               48
    table_sum               36
    table_min               27
    exp                      5

=== dev (883 entries) ===
  Valid:      883
  Invalid:    0
  No program: 0
  Operations (10 unique):
    divide                 636
    subtract               416
    add                    180
    multiply                83
    table_average           19
    greater                 10
    table_max                8
    table_min                5
    table_sum                4
    exp                      1

=== test (1147 entries) ===
  Valid:      1147
  Invalid:    0
  No program: 0
  Operations (10 unique):
    divide                 820
    subtract               521
   

## Dev.json — Detailed program analysis

Per-entry breakdown: operations used, number of steps, validity, and examples per operation.

In [16]:
OP_RE = re.compile(r"([a-z_]+)\s*\(")
dev = datasets["dev"]

# --- Per-entry analysis ---
op_counts = Counter()          # total usage of each operation
op_entries = Counter()         # number of entries that use each operation
step_counts = Counter()        # distribution of number of steps
valid = 0
invalid = 0
invalid_entries = []
op_examples = {}               # first 2 examples per operation

for entry in dev:
    prog_str = entry["qa"].get("program", "")
    if not prog_str:
        continue

    # Extract operations
    ops = OP_RE.findall(prog_str)
    num_steps = len(ops)
    step_counts[num_steps] += 1
    for op in ops:
        op_counts[op] += 1
    for op in set(ops):  # count each entry once per operation
        op_entries[op] += 1
        if op not in op_examples:
            op_examples[op] = []
        if len(op_examples[op]) < 2:
            op_examples[op].append({
                "id": entry["id"],
                "question": entry["qa"]["question"],
                "program": prog_str,
            })

    # Validity check
    try:
        tokens = parse_program_to_tokens(prog_str)
        inv_flag, result = eval_program(tokens, entry.get("table", []))
        if inv_flag:
            invalid += 1
            invalid_entries.append({"id": entry["id"], "program": prog_str, "reason": "executor_invalid"})
        else:
            valid += 1
    except Exception as e:
        invalid += 1
        invalid_entries.append({"id": entry["id"], "program": prog_str, "reason": str(e)})

# --- Print results ---
print(f"=== dev.json: {len(dev)} entries ===\n")
print(f"Valid programs:   {valid} ({valid/len(dev):.1%})")
print(f"Invalid programs: {invalid} ({invalid/len(dev):.1%})")

print(f"\n--- Step count distribution ---")
for steps in sorted(step_counts):
    c = step_counts[steps]
    bar = "#" * (c // 5)
    print(f"  {steps} step(s): {c:>4} ({c/len(dev):>5.1%})  {bar}")

print(f"\n--- Operations (total usage / entries using it) ---")
print(f"  {'Operation':<20s} {'Uses':>6} {'Entries':>8} {'% entries':>10}")
print(f"  {'-'*48}")
for op, c in op_counts.most_common():
    e = op_entries[op]
    print(f"  {op:<20s} {c:>6} {e:>8} {e/len(dev):>9.1%}")

print(f"\n--- Examples per operation ---")
for op, c in op_counts.most_common():
    print(f"\n  [{op}] ({c} uses in {op_entries[op]} entries)")
    for ex in op_examples[op]:
        print(f"    {ex['id']}")
        print(f"      Q: {ex['question'][:100]}")
        print(f"      P: {ex['program']}")

if invalid_entries:
    print(f"\n--- Invalid programs ({len(invalid_entries)}) ---")
    for ie in invalid_entries[:10]:
        print(f"  {ie['id']}: {ie['program'][:80]}")
        print(f"    reason: {ie['reason']}")

=== dev.json: 883 entries ===

Valid programs:   883 (100.0%)
Invalid programs: 0 (0.0%)

--- Step count distribution ---
  1 step(s):  523 (59.2%)  ########################################################################################################
  2 step(s):  287 (32.5%)  #########################################################
  3 step(s):   43 ( 4.9%)  ########
  4 step(s):   14 ( 1.6%)  ##
  5 step(s):   16 ( 1.8%)  ###

--- Operations (total usage / entries using it) ---
  Operation              Uses  Entries  % entries
  ------------------------------------------------
  divide                  636      610     69.1%
  subtract                416      379     42.9%
  add                     180      112     12.7%
  multiply                 83       76      8.6%
  table_average            19       19      2.2%
  greater                  10       10      1.1%
  table_max                 8        8      0.9%
  table_min                 5        5      0.6%
  table_sum       